# Feature engineering

**Goal:** select modeling positions and derive antimeridian-safe motion and
state features. The implementation lives in `storm_tracks.features`.

In [ ]:
from pathlib import Path
import pandas as pd
import tomllib

from storm_tracks.features import (
    MOTION_REGIME_LABELS,
    add_motion_features,
    select_agency_tracks,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
with (PROJECT_ROOT / "configs" / "pipeline.toml").open("rb") as file:
    config = tomllib.load(file)
intermediate_path = PROJECT_ROOT / "data" / "hurricane_paths_prepared.pkl"
processed_path = PROJECT_ROOT / config["paths"]["processed_data"]
prepared_tracks = pd.read_pickle(intermediate_path)

## Select positions and derive model features

In [ ]:
selected_tracks = select_agency_tracks(prepared_tracks)
selected_tracks = selected_tracks.loc[selected_tracks["use_for_modeling"]].reset_index(
    drop=True
)
modeling_tracks = add_motion_features(selected_tracks)
required = {"x_km", "y_km", "vx_km", "vy_km", "motion_regime"}
assert required.issubset(modeling_tracks.columns)
assert MOTION_REGIME_LABELS[2] == "low_motion"
processed_path.parent.mkdir(parents=True, exist_ok=True)
modeling_tracks.to_pickle(processed_path)
modeling_tracks[list(required)].head()

## Check longitude continuity

In [ ]:
largest_step = modeling_tracks.groupby("sid")["lon_unwrapped"].diff().abs().max()
print(f"Largest consecutive unwrapped-longitude step: {largest_step:.2f}°")

## Next step

Inspect the saved processed artifact with `03_null_model_baseline.ipynb`.
The canonical combined entry point remains `scripts/prepare_data.py`.